# 🏥 Diabetes RAG — Kaggle Notebook
**NLP Assignment 3**

### Pipeline Overview:
1. PubMed corpus (600 chunks) loaded from Kaggle dataset
2. Recursive + Fixed chunking (ablation study)
3. Embeddings via `all-MiniLM-L6-v2` → uploaded to Pinecone
4. Hybrid retrieval: BM25 + Semantic + RRF + CrossEncoder re-ranking
5. LLM generation via **OpenRouter** (free, multi-model, no credits)
6. LLM-as-a-Judge: Faithfulness (claim verification) + Relevancy (cosine similarity)
7. Ablation study: chunking strategies + retrieval modes

### ⚠️ Before running:
- Add your dataset via **Add Data**
- Set `PINECONE_API_KEY` and `OPENROUTER_API_KEY` in Section 1
- Enable GPU: Settings → Accelerator → GPU T4
- Get free OpenRouter key at: https://openrouter.ai (no credit card needed)


In [286]:
!pip install pinecone sentence-transformers rank-bm25 langchain tqdm requests -q
print("✅ All packages installed")


DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete


✅ All packages installed


In [287]:
import os, json, re, time, pickle, statistics
import numpy as np
import requests
from tqdm import tqdm
import random  # ← ADD THIS (important for jitter)
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pinecone import Pinecone, ServerlessSpec


print("✅ Imports successful")

✅ Imports successful


## 🔑 Section 1 — Configuration
> Fill in your two API keys below.

In [288]:
import os

# ──────────────────────────────────────────────────────────────
# ⚠️  PASTE YOUR KEYS HERE
# Pinecone : https://pinecone.io → API Keys
# OpenRouter: https://openrouter.ai → Keys  (free, no credit card)
# ──────────────────────────────────────────────────────────────
# PINECONE_API_KEY    = os.environ.get("PINECONE_API_KEY",    "")
# OPENROUTER_API_KEY  = os.environ.get("OPENROUTER_API_KEY",  "")

PINECONE_API_KEY    = "pcsk_2kcC2N_K6umK1LBNNfJvCCd6Aca2Yxx8oX4CEXPeceVgdBYqx67aQ8YtXQkkrgNVZ9HeTV"       # ← replace this
OPENROUTER_API_KEY  = "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"     # ← replace this
# ── OpenRouter endpoint ────────────────────────────────────────
OPENROUTER_URL      = "https://openrouter.ai/api/v1/chat/completions"

# ── Models ─────────────────────────────────────────────────────
# Generation: Qwen3-30B — reasoning model, free on OpenRouter
# Eval      : Llama-3.1-8B — lightweight, fast for many small calls
OPENROUTER_MODEL = "mistralai/mistral-small-2603"  # ← Changed to Mistral
OPENROUTER_EVAL_MODEL = "mistralai/mistral-small-2603"  # Can use same for eval

# ── Pinecone ───────────────────────────────────────────────────
PINECONE_INDEX  = "diabetes-rag"
PINECONE_REGION = "us-east-1"
PINECONE_CLOUD  = "aws"

# ── Embedding / re-ranking models ──────────────────────────────
EMBED_MODEL_NAME    = "all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# ── Chunking / retrieval hyperparameters ───────────────────────
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 50
BM25_TOP_K    = 20
SEM_TOP_K     = 20
RRF_K         = 60
RERANK_TOP_K  = 5
UPSERT_BATCH  = 100

# ── Rate limit controls ────────────────────────────────────────
# OpenRouter free tier: ~20 req/min shared across all free models.
# Qwen3 reasoning adds latency — use 3s delay to stay safe.
# ── Rate limit controls ────────────────────────────────────────
# Section 1 — Increase these values dramatically:
# ── Rate limit controls ────────────────────────────────────────
INTER_CALL_DELAY  = 10.0   # 10 seconds between API calls
INTER_QUERY_DELAY = 10     # 10 seconds between queries
MAX_CLAIMS        = 3      # Only 1 claim per query

print("✅ Config set")
print(f"   Generation model : {OPENROUTER_MODEL}")
print(f"   Eval model       : {OPENROUTER_EVAL_MODEL}")
print(f"   Pinecone index   : {PINECONE_INDEX}")
print(f"   Max claims/query : {MAX_CLAIMS}")


✅ Config set
   Generation model : mistralai/mistral-small-2603
   Eval model       : mistralai/mistral-small-2603
   Pinecone index   : diabetes-rag
   Max claims/query : 3


## 📂 Section 2 — Load Raw Data

In [289]:
import glob

candidates = glob.glob("/kaggle/input/**/*.json", recursive=True)
print("Found JSON files:", candidates)

JSON_PATH = None
for c in candidates:
    if "diabetes" in c.lower() or "abstract" in c.lower():
        JSON_PATH = c
        break
if JSON_PATH is None and candidates:
    JSON_PATH = candidates[0]
if JSON_PATH is None:
    raise FileNotFoundError("No JSON found. Add your dataset via Add Data.")

print(f"\nUsing: {JSON_PATH}")
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_records = json.load(f)

print(f"Total records loaded : {len(raw_records)}")
print(f"Keys                 : {list(raw_records[0].keys())}")
print(f"Sample title         : {raw_records[0].get('title','N/A')[:80]}")

Found JSON files: ['/kaggle/input/datasets/khansamateen/diabetes-abstracts1/diabetes_abstracts (1).json']

Using: /kaggle/input/datasets/khansamateen/diabetes-abstracts1/diabetes_abstracts (1).json
Total records loaded : 603
Keys                 : ['pmid', 'title', 'abstract', 'year', 'journal', 'authors', 'document', 'word_count']
Sample title         : Type 2 Diabetes Mellitus and Cognitive Impairment.


## 🧹 Section 3 — Text Cleaning

In [290]:
def clean_text(text):
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_document(r):
    title    = clean_text(r.get("title", ""))
    abstract = clean_text(r.get("abstract", ""))
    return (
        f"Title: {title}\n"
        f"Authors: {r.get('authors','')}\n"
        f"Year: {r.get('year','')}\nJournal: {r.get('journal','')}\n\n"
        f"Abstract: {abstract}"
    )

cleaned_records = []
skipped = 0
for r in raw_records:
    abstract_clean = clean_text(r.get("abstract", ""))
    if len(abstract_clean.split()) < 40:
        skipped += 1
        continue
    cleaned_records.append({
        "pmid":     str(r.get("pmid", "")),
        "title":    clean_text(r.get("title", "")),
        "abstract": abstract_clean,
        "year":     str(r.get("year", "")),
        "journal":  r.get("journal", ""),
        "authors":  r.get("authors", ""),
        "document": build_document(r),
    })

print(f"Cleaned : {len(cleaned_records)} records")
print(f"Skipped : {skipped} (abstract < 40 words)")
print(f"\nSample (first 300 chars):")
print(cleaned_records[0]["document"][:300])

Cleaned : 603 records
Skipped : 0 (abstract < 40 words)

Sample (first 300 chars):
Title: Type 2 Diabetes Mellitus and Cognitive Impairment.
Authors: Damanik Johanda, Yunir Em
Year: 2021
Journal: Acta medica Indonesiana

Abstract: Type 2 diabetes mellitus (T2DM) is strongly associated with lower performance on multiple domains of cognitive function and with structural abnormalitie


## ✂️ Section 4 — Chunking (Fixed & Recursive for Ablation Study)

In [291]:
def chunk_fixed(records, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    for r in records:
        words = r["document"].split()
        step  = chunk_size - overlap
        for i, start in enumerate(range(0, len(words), step)):
            chunk_text = " ".join(words[start : start + chunk_size])
            if len(chunk_text.split()) < 30:
                continue
            chunks.append({
                "chunk_id":  f"{r['pmid']}_fixed_{i}",
                "pmid":      r["pmid"],
                "title":     r["title"],
                "year":      r["year"],
                "journal":   r["journal"],
                "authors":   r["authors"],
                "text":      chunk_text,
                "strategy":  "fixed",
                "chunk_idx": i,
            })
    return chunks

def chunk_recursive(records, max_words=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    for r in records:
        doc   = r["document"]
        paras = re.split(r"\n{2,}", doc)
        buffer, buf_words, chunk_idx = "", 0, 0

        def flush(buf, idx):
            if len(buf.split()) >= 30:
                return {
                    "chunk_id":  f"{r['pmid']}_recursive_{idx}",
                    "pmid":      r["pmid"],
                    "title":     r["title"],
                    "year":      r["year"],
                    "journal":   r["journal"],
                    "authors":   r["authors"],
                    "text":      buf.strip(),
                    "strategy":  "recursive",
                    "chunk_idx": idx,
                }
            return None

        for para in paras:
            sentences = re.split(r"(?<=[.!?])\s+", para.strip())
            for sent in sentences:
                sent_words = len(sent.split())
                if buf_words + sent_words > max_words and buf_words > 0:
                    c = flush(buffer, chunk_idx)
                    if c:
                        chunks.append(c)
                        chunk_idx += 1
                    overlap_words = buffer.split()[-overlap:] if overlap > 0 else []
                    buffer    = " ".join(overlap_words) + " " + sent
                    buf_words = len(buffer.split())
                else:
                    buffer    = (buffer + " " + sent).strip()
                    buf_words += sent_words
        c = flush(buffer, chunk_idx)
        if c:
            chunks.append(c)
    return chunks

print("Generating fixed chunks...")
fixed_chunks = chunk_fixed(cleaned_records)

print("Generating recursive chunks...")
recursive_chunks = chunk_recursive(cleaned_records)

for name, chunks in [("Fixed", fixed_chunks), ("Recursive", recursive_chunks)]:
    lengths = [len(c["text"].split()) for c in chunks]
    print(f"{name}: {len(chunks)} chunks | avg {np.mean(lengths):.0f} words | "
          f"min {min(lengths)} | max {max(lengths)}")

Generating fixed chunks...
Generating recursive chunks...
Fixed: 622 chunks | avg 218 words | min 33 | max 400
Recursive: 616 chunks | avg 219 words | min 76 | max 400


## 🔢 Section 5 — Generate Embeddings (GPU)

In [292]:
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")

main_chunks = recursive_chunks[:600]
texts       = [c["text"] for c in main_chunks]

print(f"\nGenerating embeddings for {len(main_chunks)} chunks...")
t0 = time.time()
embeddings = embedder.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
elapsed = time.time() - t0
print(f"\n✅ Done in {elapsed:.1f}s  |  Shape: {embeddings.shape}")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete
DEBUG:httpcore.connection:connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c96e2e0bcb0>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x7c96f40b92d0> server_hostname='huggingface.co' timeout=10
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c96bb5ca660>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'HEAD']>


Loading embedding model: all-MiniLM-L6-v2


DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'282'), (b'Connection', b'keep-alive'), (b'Date', b'Mon, 06 Apr 2026 17:37:48 GMT'), (b'Location', b'/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-69d3ef6c-271b6e284b03af8a0713d790;dd87e754-1946-4bb6-8d36-fa458263eb89'), (b'RateLimit', b'"resolvers";r=2999;t=144'), (b'RateLimit-Policy'

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'280'), (b'Connection', b'keep-alive'), (b'Date', b'Mon, 06 Apr 2026 17:37:49 GMT'), (b'Location', b'/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6

Embedding dimension: 384

Generating embeddings for 600 chunks...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


✅ Done in 2.0s  |  Shape: (600, 384)


## ☁️ Section 6 — Upload to Pinecone

In [293]:
if PINECONE_API_KEY == "YOUR_PINECONE_API_KEY":
    raise ValueError("Set your PINECONE_API_KEY in Section 1 first.")

pc = Pinecone(api_key=PINECONE_API_KEY)

existing = [idx.name for idx in pc.list_indexes()]
if PINECONE_INDEX not in existing:
    print(f"Creating index '{PINECONE_INDEX}'...")
    pc.create_index(
        name      = PINECONE_INDEX,
        dimension = 384,
        metric    = "cosine",
        spec      = ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    time.sleep(5)
    print("Index created.")
else:
    print(f"Index '{PINECONE_INDEX}' already exists.")

pinecone_index = pc.Index(PINECONE_INDEX)

current_count = pinecone_index.describe_index_stats()["total_vector_count"]
if current_count > 0:
    print(f"Clearing {current_count} existing vectors to prevent duplicates...")
    pinecone_index.delete(delete_all=True)
    time.sleep(3)
    print("Cleared.")

print(f"\nUploading {len(main_chunks)} vectors in batches of {UPSERT_BATCH}...")
for start in tqdm(range(0, len(main_chunks), UPSERT_BATCH)):
    batch_chunks = main_chunks[start : start + UPSERT_BATCH]
    batch_embs   = embeddings[start : start + UPSERT_BATCH]
    vectors = [
        {
            "id":     c["chunk_id"],
            "values": batch_embs[j].tolist(),
            "metadata": {
                "text":     c["text"][:900],
                "pmid":     c["pmid"],
                "title":    c["title"][:200],
                "year":     c["year"],
                "journal":  c["journal"][:100],
                "authors":  c["authors"][:200],
                "strategy": c["strategy"],
            },
        }
        for j, c in enumerate(batch_chunks)
    ]
    pinecone_index.upsert(vectors=vectors)

time.sleep(2)
stats = pinecone_index.describe_index_stats()
print(f"\n✅ Upload complete!")
print(f"   Vectors in Pinecone : {stats['total_vector_count']}")
print(f"   Expected            : {len(main_chunks)}")

Index 'diabetes-rag' already exists.
Clearing 600 existing vectors to prevent duplicates...
Cleared.

Uploading 600 vectors in batches of 100...


100%|██████████| 6/6 [00:02<00:00,  2.40it/s]



✅ Upload complete!
   Vectors in Pinecone : 600
   Expected            : 600


## 🔍 Section 7 — Build BM25 Index

In [294]:
tokenized_corpus = [c["text"].lower().split() for c in main_chunks]
bm25             = BM25Okapi(tokenized_corpus)
chunk_ids        = [c["chunk_id"] for c in main_chunks]
chunk_lookup     = {c["chunk_id"]: c for c in main_chunks}
print(f"✅ BM25 index built on {len(main_chunks)} chunks")

✅ BM25 index built on 600 chunks


## ⚡ Section 8 — Load CrossEncoder Re-ranker

In [295]:
print(f"Loading CrossEncoder: {CROSS_ENCODER_MODEL}")
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print("✅ CrossEncoder ready")

DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete
DEBUG:httpcore.connection:connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c96ba2d7560>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x7c96f40b92d0> server_hostname='huggingface.co' timeout=10
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c96ba2d62d0>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'HEAD']>


Loading CrossEncoder: cross-encoder/ms-marco-MiniLM-L-6-v2


DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'96'), (b'Connection', b'keep-alive'), (b'Date', b'Mon, 06 Apr 2026 17:37:59 GMT'), (b'Location', b'/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-69d3ef77-3ff6b987312bd14e3aedee63;c2be49fd-d680-42fe-be3e-aa0553ea8efc'), (b'RateLimit', b'"resolvers";r=2987;t=133'), (b'RateLimit-Policy', b'"fixed window";"resolvers";q=3000;w=300'), (b'cross-origin-opener-policy', b'same-origin'), (b'Referrer-Policy', b'strict-origin-when-cross-origin'), (b'Access-Control-Max-Age', b'86400'), (b'Access-Control-Allow-Origin', b'https://huggingface.co'), (b'Vary', b'Origin, Accept'), (b'Access-Control-Expose-Headers', b'X-Repo-Commit,X-Request-Id,X-Error-Code,X-Error-Message,X-Total-Count,ETag,Link,Accept-Ranges,Content-Range,X-Linked-Size,X-Li

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'96'), (b'Connection', b'keep-alive'), (b'Date', b'Mon, 06 Apr 2026 17:38:00 GMT'), (b'Location', b'/cross-encoder/ms-marco-MiniL

✅ CrossEncoder ready


## 🔁 Section 9 — Hybrid Retrieval Pipeline

In [296]:
def bm25_search(query, top_k=BM25_TOP_K):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(chunk_ids[i], float(scores[i])) for i in top_indices]

def semantic_search(query, top_k=SEM_TOP_K):
    q_emb  = embedder.encode(query, normalize_embeddings=True).tolist()
    result = pinecone_index.query(vector=q_emb, top_k=top_k, include_metadata=True)
    return [(m["id"], m["score"]) for m in result["matches"]]

def rrf_fusion(bm25_results, semantic_results, k=RRF_K):
    rrf_scores = {}
    for rank, (cid, _) in enumerate(bm25_results):
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
    for rank, (cid, _) in enumerate(semantic_results):
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def rerank(query, candidates, top_k=RERANK_TOP_K):
    pairs, valid = [], []
    for cid, rrf_score in candidates:
        chunk = chunk_lookup.get(cid)
        if chunk:
            pairs.append([query, chunk["text"]])
            valid.append((cid, rrf_score))
    if not pairs:
        return []
    ce_scores = cross_encoder.predict(pairs)
    reranked = []
    for i, (cid, rrf_score) in enumerate(valid):
        chunk = chunk_lookup[cid]
        reranked.append({**chunk,
                         "rrf_score": round(float(rrf_score), 4),
                         "ce_score":  round(float(ce_scores[i]), 4)})
    reranked.sort(key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]

def retrieve(query, mode="hybrid"):
    t0 = time.time()
    if mode == "semantic":
        candidates = semantic_search(query, top_k=BM25_TOP_K + SEM_TOP_K)
    elif mode == "bm25":
        candidates = bm25_search(query, top_k=BM25_TOP_K + SEM_TOP_K)
    else:
        bm25_res   = bm25_search(query)
        sem_res    = semantic_search(query)
        candidates = rrf_fusion(bm25_res, sem_res)
    results = rerank(query, candidates)
    return results, round(time.time() - t0, 2)

print("✅ Retrieval pipeline ready (bm25 / semantic / hybrid)")

✅ Retrieval pipeline ready (bm25 / semantic / hybrid)


## 🧪 Section 10 — Test Retrieval (No LLM needed)

In [297]:
test_query = "What is the role of metformin in type 2 diabetes treatment?"
results, t = retrieve(test_query, mode="hybrid")
print(f"Query : {test_query}")
print(f"Time  : {t:.2f}s  |  Chunks returned: {len(results)}\n")
for i, r in enumerate(results, 1):
    print(f"Rank {i} | CE: {r['ce_score']:.3f} | RRF: {r['rrf_score']:.4f}")
    print(f"  Title : {r['title'][:70]}")
    print(f"  Text  : {r['text'][:200]}...\n")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Query : What is the role of metformin in type 2 diabetes treatment?
Time  : 0.38s  |  Chunks returned: 5

Rank 1 | CE: 4.470 | RRF: 0.0272
  Title : Type-2 diabetes mellitus and cardiovascular disease.
  Text  : Title: Type-2 diabetes mellitus and cardiovascular disease. Authors: Henning Robert J
Year: 2018
Journal: Future cardiology Abstract: The global prevalence of diabetes has risen in adults from 4.7% in...

Rank 2 | CE: 2.789 | RRF: 0.0139
  Title : Early therapy for type 2 diabetes in China.
  Text  : Title: Early therapy for type 2 diabetes in China. Authors: Yang Wenying, Weng Jianping
Year: 2014
Journal: The lancet. Diabetes & endocrinology Abstract: Diabetes is a huge burden in China, where abo...

Rank 3 | CE: 2.644 | RRF: 0.0292
  Title : Prebiotics Progress Shifts in the Intestinal Microbiome That Benefits 
  Text  : Title: Prebiotics Progress Shifts in the Intestinal Microbiome That Benefits Patients with Type 2 Diabetes Mellitus. Authors: Vitetta Luis, Gorgani Nick N, V

## 🤖 Section 11 — LLM Answer Generation
> Uses **OpenRouter** free API —  format as per OpenRouter docs.
> **Generation**:  — reasoning-capable model, best free quality
> **Evaluation**:  — fast, lightweight for many eval calls
>  for RAG (deterministic grounded answers; reasoning adds latency)


In [298]:
import requests

test_resp = requests.post(
    OPENROUTER_URL,
    headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"},
    json={
        "model": OPENROUTER_MODEL,
        "messages": [{"role": "user", "content": "Say hello."}],
        "max_tokens": 20
    }
)
print(test_resp.status_code, test_resp.json())

200 {'id': 'gen-1775497082-00GuSnC2lE3GzoQKkn0V', 'object': 'chat.completion', 'created': 1775497082, 'model': 'mistralai/mistral-small-2603', 'provider': 'Mistral', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': 'Hello! 😊 How can I assist you today?', 'refusal': None, 'reasoning': None}}], 'usage': {'prompt_tokens': 18, 'completion_tokens': 13, 'total_tokens': 31, 'cost': 1.05e-05, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 1.05e-05, 'upstream_inference_prompt_cost': 2.7e-06, 'upstream_inference_completions_cost': 7.8e-06}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}}


In [299]:
# ──────────────────────────────────────────────────────────────
# SECTION 11 - MISTRAL MODEL with reasoning support
# ──────────────────────────────────────────────────────────────

from openai import OpenAI

# Initialize OpenRouter client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://kaggle.com",
        "X-Title": "Diabetes RAG - Mistral"
    }
)

# Change your model to Mistral
OPENROUTER_MODEL = "mistralai/mistral-small-2603"  # ← Changed to Mistral
OPENROUTER_EVAL_MODEL = "mistralai/mistral-small-2603"  # Can use same for eval

# Rate limiting
last_call_time = 0

def wait_for_rate_limit():
    global last_call_time
    elapsed = time.time() - last_call_time
    if elapsed < INTER_CALL_DELAY:
        wait = INTER_CALL_DELAY - elapsed
        print(f"    ⏳ Rate limit wait: {wait:.1f}s")
        time.sleep(wait)
    last_call_time = time.time()

def build_prompt(query, context_chunks):
    context = "\n\n---\n\n".join([
        f"[Source {i+1}] {c['title']} ({c.get('year','')})\n{c['text']}"
        for i, c in enumerate(context_chunks)
    ])
    return (
        "You are a medical research assistant specialising in Type 2 Diabetes. "
        "Answer the question using ONLY the provided context. "
        "Cite sources as [Source N] where relevant. "
        "If context is insufficient, say so clearly.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {query}"
    )

def generate_answer(query, context_chunks, max_tokens=400, temperature=0.2, max_retries=4, enable_reasoning=False):
    """
    Generate answer using Mistral model with optional reasoning.
    Reasoning adds latency but improves accuracy for complex medical questions.
    """
    t0 = time.time()
    prompt = build_prompt(query, context_chunks)
    
    for attempt in range(max_retries):
        try:
            wait_for_rate_limit()
            
            # Build messages
            messages = [
                {"role": "system", "content": "You are a precise medical research assistant specializing in Type 2 Diabetes. Use only the provided context."},
                {"role": "user", "content": prompt}
            ]
            
            # Create request with optional reasoning
            request_params = {
                "model": OPENROUTER_MODEL,
                "messages": messages,
                "temperature": temperature,
                "max_tokens": max_tokens
            }
            
            # Add reasoning if enabled (Mistral supports this)
            if enable_reasoning:
                request_params["reasoning"] = {"enabled": True}
            
            response = client.chat.completions.create(**request_params)
            
            answer = response.choices[0].message.content.strip()
            
            # Check if reasoning_details were returned
            reasoning = getattr(response.choices[0].message, 'reasoning_details', None)
            if reasoning and enable_reasoning:
                print(f"    🤔 Reasoning used: {str(reasoning)[:100]}...")
            
            return {
                "query": query,
                "answer": answer,
                "context_chunks": context_chunks,
                "generation_time": round(time.time() - t0, 2),
                "reasoning_used": enable_reasoning
            }
            
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg:
                wait_time = min(2 ** (attempt + 2), 60)
                print(f"  Rate limited, waiting {wait_time}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(wait_time)
            elif "reasoning" in error_msg.lower():
                # Fallback without reasoning if model doesn't support it
                print(f"  Reasoning not supported, retrying without...")
                enable_reasoning = False
                time.sleep(2)
            else:
                print(f"  Error on attempt {attempt+1}: {error_msg[:100]}")
                time.sleep(5)
    
    return {
        "query": query,
        "answer": "Error: Could not generate answer after multiple retries.",
        "context_chunks": context_chunks,
        "generation_time": round(time.time() - t0, 2),
        "reasoning_used": False
    }

def call_llm_eval(prompt, model=None, max_tokens=500, enable_reasoning=False):
    """Evaluation calls using Mistral (reasoning optional for complex evals)"""
    if model is None:
        model = OPENROUTER_EVAL_MODEL
    
    for attempt in range(5):
        try:
            wait_for_rate_limit()
            
            request_params = {
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.3,
                "max_tokens": max_tokens
            }
            
            if enable_reasoning:
                request_params["reasoning"] = {"enabled": True}
            
            response = client.chat.completions.create(**request_params)
            return response.choices[0].message.content.strip()
            
        except Exception as e:
            if "429" in str(e):
                wait_time = min(2 ** (attempt + 2), 60)
                print(f"  Eval rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                time.sleep(4)
    return ""

def extract_claims(answer):
    prompt = (
        "Extract all factual claims from this answer. "
        "Return ONLY a numbered list, one claim per line. "
        "Each claim must be a single verifiable statement.\n\n"
        f"ANSWER:\n{answer}\n\nExtract claims:"
    )
    raw = call_llm_eval(prompt, max_tokens=350)
    claims = []
    for line in raw.split("\n"):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith("-")):
            claim = re.sub(r"^[\d\-\.\)\s]+", "", line).strip()
            if len(claim) > 15:
                claims.append(claim)
    if not claims:
        sentences = re.split(r"(?<=[.!?])\s+", answer)
        claims = [s.strip() for s in sentences if len(s.strip()) > 20][:5]
    return claims[:MAX_CLAIMS] if claims else [answer[:300]]

def verify_claim(claim, context):
    prompt = (
        "You are evaluating whether a research claim is supported by context.\n"
        "Read the context carefully, then answer YES if the claim is directly OR "
        "indirectly supported, or NO if the context clearly contradicts or does not "
        "mention it at all.\n\n"
        f"CONTEXT: {context[:1000]}\n\n"
        f"CLAIM: {claim}\n\n"
        "Answer YES or NO only:"
    )
    result = call_llm_eval(prompt, max_tokens=5)
    return result.strip().upper().startswith("YES")

def evaluate_faithfulness(answer, context_chunks):
    context = " ".join(c["text"] for c in context_chunks)
    claims = extract_claims(answer)
    print(f"    Extracted {len(claims)} claims")
    verifications = []
    for claim in claims:
        supported = verify_claim(claim, context)
        verifications.append({"claim": claim, "supported": supported})
    n_supported = sum(1 for v in verifications if v["supported"])
    score = n_supported / len(verifications) if verifications else 0.0
    return {
        "faithfulness_score": round(score, 3),
        "num_claims": len(claims),
        "num_supported": n_supported,
        "verifications": verifications,
    }

def evaluate_relevancy(query, answer, n_alt=3):
    prompt = (
        "Given this answer about Type 2 Diabetes, generate exactly "
        f"{n_alt} different questions that this answer addresses. "
        f"Return ONLY the questions numbered 1 to {n_alt}, one per line.\n\n"
        f"ANSWER: {answer[:500]}\n\nGenerate {n_alt} questions:"
    )
    raw = call_llm_eval(prompt, max_tokens=200)
    alt_questions = []
    for line in raw.split("\n"):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith("-")):
            q = re.sub(r"^[\d\-\.\)\s]+", "", line).strip()
            if len(q) > 10:
                alt_questions.append(q)
    alt_questions = alt_questions[:n_alt]
    if not alt_questions:
        alt_questions = [answer[:300]]
    orig_emb = embedder.encode(query, normalize_embeddings=True)
    alt_embs = embedder.encode(alt_questions, normalize_embeddings=True)
    sims = [float(np.dot(orig_emb, ae)) for ae in alt_embs]
    return {
        "relevancy_score": round(float(np.mean(sims)), 3),
        "alt_questions": alt_questions,
        "similarities": [round(s, 3) for s in sims],
    }

def evaluate(query, answer, context_chunks):
    faith = evaluate_faithfulness(answer, context_chunks)
    relev = evaluate_relevancy(query, answer)
    return {
        "faithfulness": faith,
        "relevancy": relev,
        "overall_score": round((faith["faithfulness_score"] + relev["relevancy_score"]) / 2, 3),
    }

# Test Mistral connection
print("Testing Mistral model connection...")
try:
    test_response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[{"role": "user", "content": "Say 'Mistral ready'"}],
        max_tokens=10
    )
    print(f"✅ Mistral working! Response: {test_response.choices[0].message.content}")
    
    # Optional: Test reasoning if you want
    print("\nTesting reasoning capability...")
    test_reasoning = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[{"role": "user", "content": "How many r's in strawberry?"}],
        reasoning={"enabled": True},
        max_tokens=50
    )
    reasoning_details = getattr(test_reasoning.choices[0].message, 'reasoning_details', None)
    if reasoning_details:
        print(f"✅ Reasoning supported! Preview: {str(reasoning_details)[:100]}...")
    else:
        print("⚠️ Reasoning not available (will work without it)")
        
except Exception as e:
    print(f"❌ Mistral connection failed: {e}")

print(f"\n✅ LLM generator ready (Mistral model)")
print(f"   Model: {OPENROUTER_MODEL}")
print(f"   Rate limit delay: {INTER_CALL_DELAY}s between calls")
print(f"   Max claims per query: {MAX_CLAIMS}")
print(f"   Reasoning: Optional (enable with enable_reasoning=True)")

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-981a0198-649d-463a-85ab-cc2e81f828cd', 'content': None, 'json_data': {'messages': [{'role': 'user', 'content': "Say 'Mistral ready'"}], 'model': 'mistralai/mistral-small-2603', 'max_tokens': 10}}
DEBUG:openai._base_client:Sending HTTP Request: POST https://openrouter.ai/api/v1/chat/completions
DEBUG:httpcore.connection:connect_tcp.started host='openrouter.ai' port=443 local_address=None timeout=5.0 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c96e2fb9c10>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x7c96e1634250> server_hostname='openrouter.ai' timeout=5.0
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c96e2fbb830>
DEBUG:httpcore.http11:send_request_headers.sta

Testing Mistral model connection...


DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Mon, 06 Apr 2026 17:38:03 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Access-Control-Allow-Origin', b'*'), (b'X-Generation-Id', b'gen-1775497082-kXNEZw9pjuzRjQNmO2F6'), (b'Permissions-Policy', b'payment=(self "https://checkout.stripe.com" "https://connect-js.stripe.com" "https://js.stripe.com" "https://*.js.stripe.com" "https://hooks.stripe.com")'), (b'Referrer-Policy', b'no-referrer, strict-origin-when-cross-origin'), (b'X-Content-Type-Options', b'nosniff'), (b'Content-Encoding', b'gzip'), (b'Server', b'cloudflare'), (b'CF-RAY', b'9e82905f8e1545f5-ORD')])
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:httpcore.http11:receive_response_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:receive_response_body.complete
DEBUG:httpcore.http11:respon

✅ Mistral working! Response: Mistral ready

Testing reasoning capability...
❌ Mistral connection failed: Completions.create() got an unexpected keyword argument 'reasoning'

✅ LLM generator ready (Mistral model)
   Model: mistralai/mistral-small-2603
   Rate limit delay: 10.0s between calls
   Max claims per query: 3
   Reasoning: Optional (enable with enable_reasoning=True)


## 📊 Section 12 — Evaluation: LLM-as-a-Judge
> Faithfulness = claim extraction + LLM verification
> Relevancy = alt question generation + embedding cosine similarity
> Uses lightweight `llama-3.1-8b-instruct:free` for eval (faster, saves rate limit budget)


In [300]:
def call_llm_eval(prompt, model=OPENROUTER_EVAL_MODEL, max_tokens=500):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://kaggle.com",
        "X-Title": "Diabetes RAG NLP Assignment",
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "reasoning": {"enabled": False},
    }
    
    for attempt in range(6):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers,
                                 data=_json.dumps(payload), timeout=90)
            
            if resp.status_code == 429:
                retry_after = resp.headers.get("retry-after", None)
                wait = float(retry_after) if retry_after else min(30 * (attempt + 1), 120)
                print(f"  Rate limited, waiting {wait:.0f}s (attempt {attempt+1})...")
                time.sleep(wait)
                continue
            
            if resp.status_code == 200:
                data = resp.json()
                if "error" in data:
                    print(f"  API error: {data['error']}")
                    time.sleep(15)
                    continue
                msg = data['choices'][0]['message']
                content = msg.get('content') or ''
                if content.strip():
                    time.sleep(INTER_CALL_DELAY)
                    return content.strip()
                else:
                    print(f"  Empty response on attempt {attempt+1}, retrying...")
                    time.sleep(10)
                    
        except Exception as e:
            print(f"  Error: {e}, retrying...")
            time.sleep(10)
    
    return ""

def extract_claims(answer):
    prompt = (
        "Extract all factual claims from this answer. "
        "Return ONLY a numbered list, one claim per line. "
        "Each claim must be a single verifiable statement.\n\n"
        f"ANSWER:\n{answer}\n\nExtract claims:"
    )
    raw    = call_llm_eval(prompt, max_tokens=350)
    claims = []
    for line in raw.split("\n"):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith("-")):
            claim = re.sub(r"^[\d\-\.\)\s]+", "", line).strip()
            if len(claim) > 15:
                claims.append(claim)
    if not claims:
        sentences = re.split(r"(?<=[.!?])\s+", answer)
        claims = [s.strip() for s in sentences if len(s.strip()) > 20][:5]
    # Cap to avoid excessive API calls
    return claims[:MAX_CLAIMS] if claims else [answer[:300]]

def verify_claim(claim, context):
    prompt = (
        "You are evaluating whether a research claim is supported by context.\n"
        "Read the context carefully, then answer YES if the claim is directly OR "
        "indirectly supported, or NO if the context clearly contradicts or does not "
        "mention it at all.\n\n"
        f"CONTEXT: {context[:1000]}\n\n"
        f"CLAIM: {claim}\n\n"
        "Answer YES or NO only:"
    )
    result = call_llm_eval(prompt, max_tokens=5)
    return result.strip().upper().startswith("YES")

def evaluate_faithfulness(answer, context_chunks):
    context = " ".join(c["text"] for c in context_chunks)
    claims  = extract_claims(answer)
    print(f"    Extracted {len(claims)} claims (capped at {MAX_CLAIMS})")
    verifications = []
    for claim in claims:
        supported = verify_claim(claim, context)
        verifications.append({"claim": claim, "supported": supported})
    n_supported = sum(1 for v in verifications if v["supported"])
    score = n_supported / len(verifications) if verifications else 0.0
    return {
        "faithfulness_score": round(score, 3),
        "num_claims":         len(claims),
        "num_supported":      n_supported,
        "verifications":      verifications,
    }

def evaluate_relevancy(query, answer, n_alt=3):
    prompt = (
        "Given this answer about Type 2 Diabetes, generate exactly "
        f"{n_alt} different questions that this answer addresses. "
        f"Return ONLY the questions numbered 1 to {n_alt}, one per line.\n\n"
        f"ANSWER: {answer[:500]}\n\nGenerate {n_alt} questions:"
    )
    raw = call_llm_eval(prompt, max_tokens=200)
    alt_questions = []
    for line in raw.split("\n"):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith("-")):
            q = re.sub(r"^[\d\-\.\)\s]+", "", line).strip()
            if len(q) > 10:
                alt_questions.append(q)
    alt_questions = alt_questions[:n_alt]
    method = "llm_alt_questions"
    if not alt_questions:
        alt_questions = [answer[:300]]
        method = "embedding_fallback"
    orig_emb = embedder.encode(query,         normalize_embeddings=True)
    alt_embs = embedder.encode(alt_questions, normalize_embeddings=True)
    sims     = [float(np.dot(orig_emb, ae)) for ae in alt_embs]
    return {
        "relevancy_score": round(float(np.mean(sims)), 3),
        "alt_questions":   alt_questions,
        "similarities":    [round(s, 3) for s in sims],
        "method":          method,
    }

def evaluate(query, answer, context_chunks):
    faith = evaluate_faithfulness(answer, context_chunks)
    relev = evaluate_relevancy(query, answer)
    return {
        "faithfulness":  faith,
        "relevancy":     relev,
        "overall_score": round((faith["faithfulness_score"] + relev["relevancy_score"]) / 2, 3),
    }

print("✅ Evaluation functions ready")
print(f"   Judge model : {OPENROUTER_EVAL_MODEL}")
print(f"   Claims cap  : {MAX_CLAIMS} per query")


✅ Evaluation functions ready
   Judge model : mistralai/mistral-small-2603
   Claims cap  : 3 per query


In [301]:
import logging

logging.basicConfig(level=logging.DEBUG, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

logger.debug("Debug logging is enabled")

DEBUG:__main__:Debug logging is enabled


## 🚀 Section 13 — Full End-to-End Test (10 Queries)

In [302]:
TEST_QUERIES = [
    "What is the role of metformin in type 2 diabetes treatment?",
    "How does insulin resistance develop in type 2 diabetes?",
    "What dietary interventions help manage blood glucose in diabetics?",
    "What are the cardiovascular complications of type 2 diabetes?",
    "How effective is bariatric surgery for type 2 diabetes remission?",
    "What is the relationship between obesity and type 2 diabetes?",
    "How do GLP-1 receptor agonists work in diabetes management?",
    "What role does the pancreas play in diabetes?",
    "What are the long-term complications of poorly controlled diabetes?",
    "How does physical exercise improve insulin sensitivity?",
]

all_results = []

for qi, query in enumerate(TEST_QUERIES, 1):
    # Add delay before each query (except first)
    if qi > 1:
        print(f"\n  ⏳ Waiting {INTER_QUERY_DELAY}s before next query...")
        time.sleep(INTER_QUERY_DELAY)
    
    print(f"\n{'='*60}")
    print(f"Query {qi}/{len(TEST_QUERIES)}: {query}")

    chunks, ret_time = retrieve(query, mode="hybrid")
    print(f"  Retrieved {len(chunks)} chunks in {ret_time}s")

    gen      = generate_answer(query, chunks)
    gen_time = gen["generation_time"]
    print(f"  Answer ({gen_time}s): {gen['answer'][:180].replace(chr(10),' ')}...")

    print("    → Faithfulness...")
    t_ev = time.time()
    ev   = evaluate(query, gen["answer"], chunks)
    eval_time = round(time.time() - t_ev, 2)

    print(f"  Faithfulness : {ev['faithfulness']['faithfulness_score']:.3f}  "
          f"({ev['faithfulness']['num_supported']}/{ev['faithfulness']['num_claims']} claims supported)")
    print(f"  Relevancy    : {ev['relevancy']['relevancy_score']:.3f}  "
          f"[method: {ev['relevancy'].get('method','?')}]")
    print(f"  Overall      : {ev['overall_score']:.3f}  (eval: {eval_time}s)")

    all_results.append({
        "query_id":        qi,
        "query":           query,
        "answer":          gen["answer"],
        "chunk_ids":       [c["chunk_id"] for c in chunks],
        "retrieval_time":  ret_time,
        "generation_time": gen_time,
        "evaluation_time": eval_time,
        "total_time":      round(ret_time + gen_time + eval_time, 2),
        "faithfulness":    ev["faithfulness"],
        "relevancy":       ev["relevancy"],
        "overall_score":   ev["overall_score"],
    })

    if qi < len(TEST_QUERIES):
        print(f"  ⏳ Waiting {INTER_QUERY_DELAY}s before next query...")
        time.sleep(INTER_QUERY_DELAY)

OUT_PATH = "/kaggle/working/rag_results_hybrid.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f"\n✅ Results saved → {OUT_PATH}")



Query 1/10: What is the role of metformin in type 2 diabetes treatment?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-75bdeb94-9d5a-4e9f-a182-5d16fd973e20', 'content': None, 'json_data': {'messages': [{'role': 'system', 'content': 'You are a precise medical research assistant specializing in Type 2 Diabetes. Use only the provided context.'}, {'role': 'user', 'content': 'You are a medical research assistant specialising in Type 2 Diabetes. Answer the question using ONLY the provided context. Cite sources as [Source N] where relevant. If context is insufficient, say so clearly.\n\nCONTEXT:\n[Source 1] Type-2 diabetes mellitus and cardiovascular disease. (2018)\nTitle: Type-2 diabetes mellitus and cardiovascular disease. Authors: Henning Robert J\nYear: 2018\nJournal: Future cardiology Abstract: The global prevalence of diabetes has risen in adults from 4.7% in 1980 to 8.5% in 2014. 90-95% of adults with diabetes have Type 2 diabetes (T2D). This paper focuses

  Retrieved 5 chunks in 0.32s


...

DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Mon, 06 Apr 2026 17:50:35 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Access-Control-Allow-Origin', b'*'), (b'X-Generation-Id', b'gen-1775497834-Fv3cNCxsuGIKDftEFPAU'), (b'Permissions-Policy', b'payment=(self "https://checkout.stripe.com" "https://connect-js.stripe.com" "https://js.stripe.com" "https://*.js.stripe.com" "https://hooks.stripe.com")'), (b'Referrer-Policy', b'no-referrer, strict-origin-when-cross-origin'), (b'X-Content-Type-Options', b'nosniff'), (b'Content-Encoding', b'gzip'), (b'Server', b'cloudflare'), (b'CF-RAY', b'9e82a2b82efd9327-ORD')])
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:httpcore.http11:receive_response_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:receive_response_body.complete
DEBUG:httpcore.http11:respon

  Answer (3.1s): Physical exercise improves insulin sensitivity through several mechanisms as supported by the provided context:  1. **Enhancement of Insulin Secretion and Islet Function**: Chronic...
    → Faithfulness...
    Extracted 3 claims (capped at 3)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Faithfulness : 0.333  (1/3 claims supported)
  Relevancy    : 0.786  [method: llm_alt_questions]
  Overall      : 0.559  (eval: 55.35s)

✅ Results saved → /kaggle/working/rag_results_hybrid.json


In [303]:
# ✅ NOTE: The throttling/backoff logic is now built directly into
# call_openrouter() in Section 11. This cell is intentionally left
# as a no-op placeholder to preserve cell numbering.
#
# OpenRouter rate limit strategy:
#   - INTER_CALL_DELAY = 2.0s sleep after every successful call (Section 1)
#   - MAX_CLAIMS = 5 cap per query (Section 1) → max 7 calls per query
#   - INTER_QUERY_DELAY = 10s between queries (Section 13 loop)
#   - 429 retry with exponential backoff built into call_openrouter()
#   - Eval calls use llama-3.1-8b:free (faster, lower token count)
print("✅ Rate limit strategy: see call_openrouter() in Section 11")
print(f"   INTER_CALL_DELAY  = {INTER_CALL_DELAY}s")
print(f"   INTER_QUERY_DELAY = {INTER_QUERY_DELAY}s")
print(f"   MAX_CLAIMS        = {MAX_CLAIMS}")
print(f"   Gen model         : {OPENROUTER_MODEL}")
print(f"   Eval model        : {OPENROUTER_EVAL_MODEL}")


✅ Rate limit strategy: see call_openrouter() in Section 11
   INTER_CALL_DELAY  = 10.0s
   INTER_QUERY_DELAY = 10s
   MAX_CLAIMS        = 3
   Gen model         : mistralai/mistral-small-2603
   Eval model        : mistralai/mistral-small-2603


## 📐 Section 14 — Ablation Study

In [304]:
def chunk_stats(chunks, name):
    lengths = [len(c["text"].split()) for c in chunks]
    return {
        "strategy":   name,
        "num_chunks": len(chunks),
        "avg_words":  round(float(np.mean(lengths)), 1),
        "min_words":  min(lengths),
        "max_words":  max(lengths),
        "std_words":  round(float(np.std(lengths)), 1),
    }

ablation_chunking = [
    chunk_stats(fixed_chunks,     "Fixed (400w, 50 overlap)"),
    chunk_stats(recursive_chunks, "Recursive (400w, 50 overlap)"),
]

print("=== ABLATION 1: Chunking Strategy ===")
print(f"{'Strategy':<35} {'Chunks':>7} {'Avg':>6} {'Min':>5} {'Max':>5} {'Std':>6}")
print("-" * 65)
for s in ablation_chunking:
    print(f"{s['strategy']:<35} {s['num_chunks']:>7} {s['avg_words']:>6.1f} "
          f"{s['min_words']:>5} {s['max_words']:>5} {s['std_words']:>6.1f}")

# ── ABLATION 2: Retrieval Mode ──────────────────────────────────────────────
# Strategy: reuse CACHED answers from Section 13 (all_results) for hybrid mode.
# Only re-retrieve + re-generate for bm25 and semantic modes (6 queries total).
# This saves ~60 API calls and avoids rate-limit stalls.

ABLATION_QUERIES = TEST_QUERIES[:3]
MODES = ["bm25", "semantic", "hybrid"]

# Build a lookup of cached hybrid answers from all_results
hybrid_cache = {r["query"]: r for r in all_results if r["query"] in ABLATION_QUERIES}

print("\n=== ABLATION 2: Retrieval Mode ===")
print(f"  Cached hybrid results available: {len(hybrid_cache)}/3")
print("  BM25 + Semantic: will retrieve + generate fresh (6 queries × ~5 LLM calls each)")
print("  Hybrid: reusing Section 13 cached results — no extra API calls")

ablation_retrieval = []

for mode in MODES:
    print(f"\n  Mode: {mode.upper()}")
    mode_row = {"mode": mode, "queries": []}

    for qi, query in enumerate(ABLATION_QUERIES):
        print(f"    Query {qi+1}/3: {query[:60]}...")

        if mode == "hybrid" and query in hybrid_cache:
            # ✅ Reuse cached result — zero extra API calls
            cached = hybrid_cache[query]
            mode_row["queries"].append({
                "query":          query,
                "retrieval_time": cached["retrieval_time"],
                "faithfulness":   cached["faithfulness"]["faithfulness_score"],
                "relevancy":      cached["relevancy"]["relevancy_score"],
                "overall":        cached["overall_score"],
                "source":         "cached",
            })
            print(f"      (cached) Faith={cached['faithfulness']['faithfulness_score']:.3f}  "
                  f"Relev={cached['relevancy']['relevancy_score']:.3f}  "
                  f"Overall={cached['overall_score']:.3f}")
        else:
            # Fresh retrieval + generation for bm25 / semantic
            chunks, ret_time = retrieve(query, mode=mode)
            gen = generate_answer(query, chunks, max_tokens=300)
            answer = gen["answer"]
            ev = evaluate(query, answer, chunks)
            
            mode_row["queries"].append({
                "query":          query,
                "retrieval_time": ret_time,
                "faithfulness":   ev["faithfulness"]["faithfulness_score"],
                "relevancy":      ev["relevancy"]["relevancy_score"],
                "overall":        ev["overall_score"],
                "source":         "fresh",
            })
            print(f"      Faith={ev['faithfulness']['faithfulness_score']:.3f}  "
                  f"Relev={ev['relevancy']['relevancy_score']:.3f}  "
                  f"Overall={ev['overall_score']:.3f}")

        if qi < len(ABLATION_QUERIES) - 1:
            print(f"      ⏳ Waiting {INTER_QUERY_DELAY}s...")
            time.sleep(INTER_QUERY_DELAY)

    ablation_retrieval.append(mode_row)

    if mode != MODES[-1]:
        print(f"  ⏳ Waiting 20s before next mode...")
        time.sleep(20)

# ── Summary table ────────────────────────────────────────────────────────────
print(f"\n{'Mode':<12} {'Avg Faith':>10} {'Avg Relev':>10} {'Avg Overall':>12} {'Avg Ret(s)':>10}")
print("-" * 58)
for m in ablation_retrieval:
    qs = m["queries"]
    if qs:
        print(f"{m['mode']:<12} "
              f"{np.mean([q['faithfulness'] for q in qs]):>10.3f} "
              f"{np.mean([q['relevancy'] for q in qs]):>10.3f} "
              f"{np.mean([q['overall'] for q in qs]):>12.3f} "
              f"{np.mean([q['retrieval_time'] for q in qs]):>10.2f}")

ablation_out = {"chunking_ablation": ablation_chunking, "retrieval_ablation": ablation_retrieval}
with open("/kaggle/working/ablation_results.json", "w") as f:
    json.dump(ablation_out, f, indent=2)
print("\n✅ Ablation results saved → /kaggle/working/ablation_results.json")

=== ABLATION 1: Chunking Strategy ===
Strategy                             Chunks    Avg   Min   Max    Std
-----------------------------------------------------------------
Fixed (400w, 50 overlap)                622  217.6    33   400   70.2
Recursive (400w, 50 overlap)            616  219.4    76   400   66.9

=== ABLATION 2: Retrieval Mode ===
  Cached hybrid results available: 3/3
  BM25 + Semantic: will retrieve + generate fresh (6 queries × ~5 LLM calls each)
  Hybrid: reusing Section 13 cached results — no extra API calls

  Mode: BM25
    Query 1/3: What is the role of metformin in type 2 diabetes treatment?...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-6aa90d36-491e-49c2-9a6f-71bbaaffde2b', 'content': None, 'json_data': {'messages': [{'role': 'system', 'content': 'You are a precise medical research assistant specializing in Type 2 Diabetes. Use only the provided context.'}, {'role': 'user', 'content': 'You are a medical research assistant specialising in Type 2 Diabetes. Answer the question using ONLY the provided context. Cite sources as [Source N] where relevant. If context is insufficient, say so clearly.\n\nCONTEXT:\n[Source 1] Type-2 diabetes mellitus and cardiovascular disease. (2018)\nTitle: Type-2 diabetes mellitus and cardiovascular disease. Authors: Henning Robert J\nYear: 2018\nJournal: Future cardiology Abstract: The global prevalence of diabetes has risen in adults from 4.7% in 1980 to 8.5% in 2014. 90-95% of adults with diabetes have Type 2 diabetes (T2D). This paper focuses

    Extracted 3 claims (capped at 3)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-083360ee-af47-442b-9c21-cd0c614f3b59', 'content': None, 'json_data': {'messages': [{'role': 'system', 'content': 'You are a precise medical research assistant specializing in Type 2 Diabetes. Use only the provided context.'}, {'role': 'user', 'content': 'You are a medical research assistant specialising in Type 2 Diabetes. Answer the question using ONLY the provided context. Cite sources as [Source N] where relevant. If context is insufficient, say so clearly.\n\nCONTEXT:\n[Source 1] Pharmacological regulation of blood glucose levels in non-insulin-dependent diabetes mellitus. (1997)\nTitle: Pharmacological regulation of blood glucose levels in non-insulin-dependent diabetes mellitus. Authors: Bressler R, Johnson D G\nYear: 1997\nJournal: Archives of internal medicine Abstract: Non-insulin-dependent diabetes mellitus is a metabolic disease 

    Extracted 3 claims (capped at 3)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

      Faith=1.000  Relev=0.711  Overall=0.855
  ⏳ Waiting 20s before next mode...

  Mode: HYBRID
    Query 1/3: What is the role of metformin in type 2 diabetes treatment?...
      (cached) Faith=0.667  Relev=0.792  Overall=0.730
      ⏳ Waiting 10s...
    Query 2/3: How does insulin resistance develop in type 2 diabetes?...
      (cached) Faith=1.000  Relev=0.795  Overall=0.897
      ⏳ Waiting 10s...
    Query 3/3: What dietary interventions help manage blood glucose in diab...
      (cached) Faith=1.000  Relev=0.695  Overall=0.847

Mode          Avg Faith  Avg Relev  Avg Overall Avg Ret(s)
----------------------------------------------------------
bm25              1.000      0.773        0.887       0.32
semantic          0.889      0.789        0.839       0.62
hybrid            0.889      0.761        0.825       0.48

✅ Ablation results saved → /kaggle/working/ablation_results.json


## 📋 Section 15 — Save Artefacts & Final Summary

In [305]:
with open("/kaggle/working/bm25_index.pkl", "wb") as f:
    pickle.dump({"bm25": bm25, "chunks": main_chunks, "chunk_ids": chunk_ids}, f)

print("✅ Artefacts saved to /kaggle/working/:")
print("   bm25_index.pkl          ← BM25 index (for Streamlit app)")
print("   rag_results_hybrid.json ← Full 10-query evaluation")
print("   ablation_results.json   ← Chunking & retrieval ablation")

faith_scores   = [r["faithfulness"]["faithfulness_score"] for r in all_results]
relev_scores   = [r["relevancy"]["relevancy_score"]        for r in all_results]
overall_scores = [r["overall_score"]                       for r in all_results]
ret_times      = [r["retrieval_time"]                      for r in all_results]
gen_times      = [r["generation_time"]                     for r in all_results]
total_times    = [r["total_time"]                          for r in all_results]

print("\n" + "="*55)
print("FINAL EVALUATION SUMMARY")
print("="*55)
print(f"  Queries evaluated    : {len(all_results)}")
print(f"  LLM (generation)     : {OPENROUTER_MODEL}")
print(f"  Avg Faithfulness     : {statistics.mean(faith_scores):.3f}")
print(f"  Avg Relevancy        : {statistics.mean(relev_scores):.3f}")
print(f"  Avg Overall          : {statistics.mean(overall_scores):.3f}")
print(f"  Avg Retrieval Time   : {statistics.mean(ret_times):.2f}s")
print(f"  Avg Generation Time  : {statistics.mean(gen_times):.2f}s")
print(f"  Avg Total Time/Query : {statistics.mean(total_times):.1f}s")
print(f"  Chunks in Pinecone   : {len(main_chunks)}")
print(f"  Chunking strategy    : Recursive (primary)")
print(f"  Retrieval mode       : Hybrid (BM25 + Semantic + RRF + CrossEncoder)")
print(f"  Embedding model      : {EMBED_MODEL_NAME} (384-dim)")
print()
print(f"  {'Q':>3}  {'Faith':>6}  {'Relev':>6}  {'Overall':>8}  {'Time':>6}")
print("  " + "-"*38)
for r in all_results:
    print(f"  Q{r['query_id']:<2}  "
          f"{r['faithfulness']['faithfulness_score']:>6.3f}  "
          f"{r['relevancy']['relevancy_score']:>6.3f}  "
          f"{r['overall_score']:>8.3f}  "
          f"{r['total_time']:>5.1f}s")

✅ Artefacts saved to /kaggle/working/:
   bm25_index.pkl          ← BM25 index (for Streamlit app)
   rag_results_hybrid.json ← Full 10-query evaluation
   ablation_results.json   ← Chunking & retrieval ablation

FINAL EVALUATION SUMMARY
  Queries evaluated    : 10
  LLM (generation)     : mistralai/mistral-small-2603
  Avg Faithfulness     : 0.900
  Avg Relevancy        : 0.775
  Avg Overall          : 0.837
  Avg Retrieval Time   : 0.55s
  Avg Generation Time  : 4.09s
  Avg Total Time/Query : 62.9s
  Chunks in Pinecone   : 600
  Chunking strategy    : Recursive (primary)
  Retrieval mode       : Hybrid (BM25 + Semantic + RRF + CrossEncoder)
  Embedding model      : all-MiniLM-L6-v2 (384-dim)

    Q   Faith   Relev   Overall    Time
  --------------------------------------
  Q1    0.667   0.792     0.730   55.8s
  Q2    1.000   0.795     0.897   61.9s
  Q3    1.000   0.695     0.847   62.6s
  Q4    1.000   0.831     0.915   68.8s
  Q5    1.000   0.826     0.913   61.0s
  Q6    1.000  